In [ ]:
import uproot
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import sys
import yaml

from ipywidgets import interact

In [ ]:
SOFTWARE_DIR = '/global/homes/k/kjvocker/spine/src'  # Path to software install

TMS_DIR = "/global/cfs/cdirs/dunepro/people/abooth/nd-production/output/MiniProdN5/run-tms-reco/MiniProdN5p1_NDComplex_FHC.tmsreco.full.sanddrift/TMSRECO/0002000/"
NDLAr_DIR = "/global/cfs/cdirs/dunepro/people/abooth/nd-production/output/MiniProdN5/run-mlreco/MiniProdN5p1_NDComplex_FHC.spine.full.sanddrift/MLRECO_SPINE/0002000/"

filenum = 471

TMS_file = f"{TMS_DIR}MiniProdN5p1_NDComplex_FHC.tmsreco.full.sanddrift.0002{filenum}.TMSRECO.root"
SPINE_file = f"{NDLAr_DIR}MiniProdN5p1_NDComplex_FHC.spine.full.sanddrift.0002{filenum}.MLRECO_SPINE.hdf5"

# Set software directory
sys.path.append(SOFTWARE_DIR)
from spine.driver import Driver

## Colour Palette

In [ ]:
NDLAr_colour = '#e63946'

# List 1: Accessible Base (Strong & Distinct)
strong_colours = [
    '#E69F00', # Orange
    '#0072B2', # Deep Blue
    '#009E73', # Bluish Green
    '#CC79A7', # Reddish Purple
    '#D55E00', # Vermillion
    '#56B4E9', # Sky Blue
    '#F0E442'  # Yellow
]

# List 2: Accessible Tint (Lighter "Shadow" Versions)
light_colours = [
    '#f0c975', # Light Orange
    '#70b1d9', # Pale Blue
    '#7ecdb9', # Light Green
    '#e0adc9', # Light Orchid
    '#e8a176', # Light Salmon
    '#a2d9f7', # Very Light Blue
    '#f7f2a1'  # Pale Yellow
]

## Create ND-LAr and TMS Geometry

In [ ]:
# Geometry defined in TMS_Constants.h:

#  // Approximate starting and end positions of TMS detector in geometry for plotting hits, in {x,y,z}
#  // in mm by default!
#  const double TMS_Start[] = {-4000, -3500, 11000};
#  const double TMS_End[] = {4000, 500, 19000};

# const double TMS_Double_End = 18502.5;  //this is still with problems in the geometry!!!
# const double TMS_Thin_Start = 11177.5;  //start of TMS at 1116 cm - bar thickness / 2 (8 mm) = 11160 - 8 = 11152 mm
# const double TMS_Start_Exact[] = {-3730, -3702.23, TMS_Thin_Start};
# const double TMS_End_Exact[] = {3730, 997.77, TMS_Double_End};

#  // From scanning all hits x,y,z position in the LAr active volume
#  const double LAr_Start_Exact[] = {-3478.48, -2166.71, 4179.24};
#  const double LAr_End_Exact[] = {3478.48, 829.282, 9135.88};

In [ ]:
def plot_box(fig, x_min, x_max, y_min, y_max, z_min, z_max, color, name):
    # This path visits every corner of a box in one continuous line.
    # We use 'None' to jump between disconnected edges.
    x = [x_min, x_max, x_max, x_min, x_min, x_min, x_max, x_max, x_min, x_min, None, x_max, x_max, None, x_min, x_min, None, x_max, x_max]
    y = [y_min, y_min, y_max, y_max, y_min, y_min, y_min, y_max, y_max, y_min, None, y_min, y_min, None, y_max, y_max, None, y_max, y_max]
    z = [z_min, z_min, z_min, z_min, z_min, z_max, z_max, z_max, z_max, z_max, None, z_min, z_max, None, z_min, z_max, None, z_min, z_max]

    fig.add_trace(
        go.Scatter3d(
            x=x, y=y, z=z,
            mode="lines",
            line=dict(color=color, width=3),
            name=name,
            showlegend=True
        )
    )

## Class to pull data from TMS file

In [ ]:
class TMSData:
    def __init__(self, filename):
        self.file = uproot.open(filename)

        # Truth Info
        self.EventNo = self.file["Truth_Info"]["EventNo"].array(library="np")
        self.SpillNo = self.file["Truth_Info"]["SpillNo"].array(library="np")
        self.RunNo = self.file["Truth_Info"]["RunNo"].array(library="np")
        self.Interation = self.file["Truth_Info"]["Interaction"].array(library="np")

        self.TruthInfoIndex = self.file["Truth_Info"]["TruthInfoIndex"].array(library="np")
        self.TruthInfoNSlices = self.file["Truth_Info"]["TruthInfoNSlices"].array(library="np")
        self.nPrimaryVertices = self.file["Truth_Info"]["nPrimaryVertices"].array(library="np")
        self.nParticles = self.file["Truth_Info"]["nParticles"].array(library="np")
        self.MuonVertex = self.file["Truth_Info"]["Muon_Vertex"].array(library="np")
        self.VertexIdOfMostEnergyInEvent = self.file["Truth_Info"]["VertexIdOfMostEnergyInEvent"].array(library="np")
        self.RecoTrackN = self.file["Truth_Info"]["RecoTrackN"].array(library="np")

        self.NeutrinoX4 = self.file["Truth_Info"]["NeutrinoX4"].array(library="np")

        self.RecoTrackPrimaryParticleIndex = self.file["Truth_Info"]["RecoTrackPrimaryParticleIndex"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionTrackStart = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionTrackStart"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionTrackEnd = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionTrackEnd"].array(library="np")
        self.RecoTrackTrueHitPosition = self.file["Truth_Info"]["RecoTrackTrueHitPosition"].array(library="np")
        self.RecoTrackPrimaryParticlePDG = self.file["Truth_Info"]["RecoTrackPrimaryParticlePDG"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionStart = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionStart"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionEnd = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionEnd"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionEnteringTMS = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionEnteringTMS"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionLeavingTMS = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionLeavingTMS"].array(library="np")
        self.RecoTrackPrimaryParticleTruePositionLeavingLAr = self.file["Truth_Info"]["RecoTrackPrimaryParticleTruePositionLeavingLAr"].array(library="np")
        self.RecoTrackPrimaryParticleVtxID = self.file["Truth_Info"]["RecoTrackPrimaryParticleVtxId"].array(library="np")

        self.RecoTrackSecondaryParticleIndex = self.file["Truth_Info"]["RecoTrackSecondaryParticleIndex"].array(library="np")
        self.RecoTrackSecondaryParticlePDG = self.file["Truth_Info"]["RecoTrackSecondaryParticlePDG"].array(library="np")
        self.RecoTrackSecondaryParticleTruePositionStart = self.file["Truth_Info"]["RecoTrackSecondaryParticleTruePositionStart"].array(library="np")
        self.RecoTrackSecondaryParticleTruePositionEnd = self.file["Truth_Info"]["RecoTrackSecondaryParticleTruePositionEnd"].array(library="np")

        self.TrueHitX = self.file["Truth_Info"]["TrueHitX"].array(library="np")
        self.TrueHitY = self.file["Truth_Info"]["TrueHitY"].array(library="np")
        self.TrueHitZ = self.file["Truth_Info"]["TrueHitZ"].array(library="np")
        self.TrueHitT = self.file["Truth_Info"]["TrueHitT"].array(library="np")
        self.TrueRecoHitX = self.file["Truth_Info"]["TrueRecoHitX"].array(library="np")
        self.TrueRecoHitY = self.file["Truth_Info"]["TrueRecoHitY"].array(library="np")
        self.TrueRecoHitZ = self.file["Truth_Info"]["TrueRecoHitZ"].array(library="np")
        self.TrueRecoHitT = self.file["Truth_Info"]["TrueRecoHitT"].array(library="np")

        self.TrueVtxN = self.file["Truth_Spill"]["TrueVtxN"].array(library="np")
        self.TrueVtxX = self.file["Truth_Spill"]["TrueVtxX"].array(library="np")
        self.TrueVtxY = self.file["Truth_Spill"]["TrueVtxY"].array(library="np")
        self.TrueVtxZ = self.file["Truth_Spill"]["TrueVtxZ"].array(library="np")
        self.TrueVtxT = self.file["Truth_Spill"]["TrueVtxT"].array(library="np")

        self.TrueNHits = self.file["Truth_Spill"]["TrueNHits"].array(library="np")
        self.PDG = self.file["Truth_Spill"]["PDG"].array(library="np")
        self.TrueVtxPDG = self.file["Truth_Spill"]["TrueVtxPDG"].array(library="np")
        self.TrueVtxID = self.file["Truth_Spill"]["TrueVtxID"].array(library="np")
        self.VertexID = self.file["Truth_Spill"]["VertexID"].array(library="np")
        self.TrackID = self.file["Truth_Spill"]["TrackId"].array(library="np")
        self.TruthInfoIndex = self.file["Truth_Spill"]["TruthInfoIndex"].array(library="np")

        self.PositionLArStart = self.file["Truth_Spill"]["PositionLArStart"].array(library="np")
        self.PositionLArEnd = self.file["Truth_Spill"]["PositionLArEnd"].array(library="np")
        self.PositionTMSStart = self.file["Truth_Spill"]["PositionTMSStart"].array(library="np")
        self.PositionTMSFirstTwoModulesEnd = self.file["Truth_Spill"]["PositionTMSFirstTwoModulesEnd"].array(library="np")
        self.PositionTMSThinEnd = self.file["Truth_Spill"]["PositionTMSThinEnd"].array(library="np")
        self.PositionTMSEnd = self.file["Truth_Spill"]["PositionTMSEnd"].array(library="np")
        self.BirthPosition = self.file["Truth_Spill"]["BirthPosition"].array(library="np")
        self.DeathPosition = self.file["Truth_Spill"]["DeathPosition"].array(library="np")

        self.EventNo_TruthSpill = self.file["Truth_Spill"]["EventNo"].array(library="np")
        self.SpillNo_TruthSpill = self.file["Truth_Spill"]["SpillNo"].array(library="np")

        # Reco Tree
        self.RECO_StartPos = self.file["Reco_Tree"]["StartPos"].array(library="np")
        self.RECO_EndPos = self.file["Reco_Tree"]["EndPos"].array(library="np")

## Setup driver to read ND-LAr info

In [ ]:
## Call files and load SPINE driver to read spine file

cfg = f'''
base:
  verbosity: warning
build:
  mode: both
  fragments: false
  particles: true
  interactions: true
  units: cm
###
post:
  match:
    particle:
      match_mode: both
    interaction:
      match_mode: both
###
io:
  reader:
    file_keys: {SPINE_file}
    skip_unknown_attrs: True
    name: hdf5
'''
driver = Driver(yaml.safe_load(cfg))

## Choose spill and load ND-LAr / TMS truth info

In [ ]:
spill = 11    # For now only spill 0 works with vertex timing in TMS

data = driver.process(spill)
truth_particles = data['truth_particles']
truth_interactions = data['truth_interactions']

print("Nof true particles in ND-LAr spill", spill, "/", len(driver), ":", len(truth_particles))
print("Nof true interactions in ND-LAr spill", spill, "/", len(driver), ":", len(truth_interactions))

In [ ]:
TMS = TMSData(TMS_file)
print("Total Truth_Info entries:", len(TMS.EventNo))
print("Looking at spill", spill, "/", TMS.SpillNo[-1])

mask = (TMS.SpillNo == spill)
TMS_indices = np.where(mask)[0]
print("Nof TMS Truth_Info entries in spill", spill, ":", len(TMS_indices))
print("Nof TMS Truth_Spill vertices in spill", spill, ":", len(TMS.TrueVtxID[spill]))

## Match TMS and SPINE truth vertices and tracks

Two-stage matching (see `track_vertex_matching.py` for the standalone version this is adapted from):

1. **Vertices** - nearest-neighbor match on true vertex position (mm): TMS `TrueVtxX/Y/Z`
   (`Truth_Spill`) vs SPINE `truth_interaction.vertex * 10` (cm -> mm).
2. **Tracks** - for each matched vertex pair, use `TrackId`/`track_id` as a fast candidate
   join key, then CONFIRM every candidate with PDG code + birth position agreement before
   accepting it. This guards against raw-id collisions across truth interactions: SPINE's own
   vertex clustering is not guaranteed to be a 1:1 mirror of TMS's true MC `VertexID`, so the
   same raw G4 track id can occasionally land in two different "matched" interactions
   pointing at two different physical particles.
3. **Hits** - TMS's Truth_Info tree has no per-hit truth track id (and `Truth_Spill`'s
   `TrueNHits` is only a ground-truth COUNT per particle, not a list of hit indices -- nothing
   in the file says *which* hits are whose), so hits are assigned to a confirmed track by
   nearest distance to that track's true trajectory *polyline* (`BirthPosition ->
   PositionLArStart/End -> PositionTMSStart/FirstTwoModulesEnd/ThinEnd/End -> DeathPosition`,
   dropping the `-1e8` sentinel legs a particle didn't reach). A single Birth->Death chord is a
   bad proxy for a track that bends in the TMS field (checked against a real muon: 6 hits
   within 100mm using the chord vs. 59 using the full polyline, against a true `TrueNHits` of
   54) -- the polyline is a much closer match, but it's still a straight-line approximation
   between waypoints, so treat the hit colouring as a visualization aid, not a hit-level truth
   match. `TrueNHits` is used as each track's assignment *quota* (see point 5) rather than
   being able to identify hits directly.
4. **Narrowing the hit search by VertexID** - rather than searching every true hit in the whole
   spill, `find_tms_slices_for_vertex` uses `VertexIdOfMostEnergyInEvent` and
   `RecoTrackPrimaryParticleVtxId` (both per `Truth_Info` slice) to find just the slice(s) that
   belong to a given TMS `VertexID`, and hits are only pulled from those. This isn't just
   faster (596 candidate hits instead of 17443 for the vertex checked) -- it's also more
   accurate: searching the whole spill let 5+ coincidentally-nearby hits from unrelated pileup
   vertices sneak into the muon's 100mm cut (59 vs true 54); scoping to the vertex's own
   slice(s) first brought that down to 52.
5. **TrueNHits as a quota, not an index** - `assign_hits_to_tracks` uses each confirmed
   track's `TrueNHits` as how many hits it's allowed to claim, then does one global greedy
   pass over every (hit, track) pair sorted by distance, honoring quotas as it goes. This
   matters when two confirmed tracks' polylines pass near the same hits: a single fixed
   distance threshold has no principled way to split contested hits between them, while a
   quota lets each track take only as many of the contested hits as its own truth count
   allows, leaving the rest for the other track.

`NDLAr_idx` (used everywhere below and by the plot slider) is the index into the
`truth_interactions` list, which SPINE assigns as `ixn.id` -- this is the same key
`TruthParticle.interaction_id` uses to link a particle back to its interaction.

In [ ]:
VERTEX_TOL_MM = 1.0        # vertex position match tolerance (mm)
TRACK_POS_TOL_MM = 0.5     # birth-position confirmation tolerance for tracks (mm)
HIT_ASSIGN_TOL_MM = 100.0  # max distance from a hit to a track's polyline to colour it as that track

SENTINEL = -1e8
# Ordered waypoints tracing a particle's true trajectory through LAr and TMS.
# Not every particle reaches every waypoint (e.g. it may stop in LAr, or never
# enter TMS) -- those legs come back as (-1e8, -1e8, -1e8) and are dropped
# when building the polyline.
WAYPOINT_FIELDS = [
    "BirthPosition", "PositionLArStart", "PositionLArEnd", "PositionTMSStart",
    "PositionTMSFirstTwoModulesEnd", "PositionTMSThinEnd", "PositionTMSEnd",
    "DeathPosition",
]


def build_polyline(waypoints):
    """Drop sentinel (-1e8) and consecutive-duplicate waypoints."""
    pts = []
    for w in waypoints:
        p = np.asarray(w[:3], dtype=float)
        if np.any(p <= SENTINEL / 2):
            continue
        if not pts or not np.allclose(p, pts[-1]):
            pts.append(p)
    return pts


def point_segment_dist(p, a, b):
    ab = b - a
    denom = np.dot(ab, ab)
    if denom == 0:
        return np.linalg.norm(p - a)
    t = np.clip(np.dot(p - a, ab) / denom, 0.0, 1.0)
    return np.linalg.norm(p - (a + t * ab))


def point_polyline_dist(p, pts):
    if len(pts) == 1:
        return np.linalg.norm(p - pts[0])
    return min(point_segment_dist(p, pts[k], pts[k + 1]) for k in range(len(pts) - 1))


def build_tms_vertex_table(TMS, spill):
    vtx_id = TMS.TrueVtxID[spill]
    vtx_x = TMS.TrueVtxX[spill]
    vtx_y = TMS.TrueVtxY[spill]
    vtx_z = TMS.TrueVtxZ[spill]
    return [
        {"vtx_id": int(vtx_id[i]), "pos": np.array([vtx_x[i], vtx_y[i], vtx_z[i]], dtype=float)}
        for i in range(len(vtx_id))
    ]


def build_tms_particle_table(TMS, spill):
    p_vtx_id = TMS.VertexID[spill]
    p_track_id = TMS.TrackID[spill]
    p_pdg = TMS.PDG[spill]
    p_birth = TMS.BirthPosition[spill]   # (N,4): x,y,z,t (mm)
    p_death = TMS.DeathPosition[spill]
    p_n_hits = TMS.TrueNHits[spill]      # ground-truth hit COUNT per particle (not a hit index list)
    waypoint_arrs = {f: getattr(TMS, f)[spill] for f in WAYPOINT_FIELDS}

    particles_by_vertex = {}
    for i in range(len(p_track_id)):
        vid = int(p_vtx_id[i])
        particles_by_vertex.setdefault(vid, []).append({
            "track_id": int(p_track_id[i]),
            "pdg": int(p_pdg[i]),
            "birth_pos": np.array(p_birth[i][:3], dtype=float),
            "death_pos": np.array(p_death[i][:3], dtype=float),
            "polyline": build_polyline([waypoint_arrs[f][i] for f in WAYPOINT_FIELDS]),
            "n_hits": int(p_n_hits[i]),
        })
    return particles_by_vertex


def build_tms_spill_hits(TMS, spill):
    """All true TMS hit positions (mm) across every Truth_Info slice in this spill."""
    idxs = np.where(TMS.SpillNo == spill)[0]
    if len(idxs) == 0:
        return np.empty((0, 3))
    hx = np.concatenate([TMS.TrueHitX[i] for i in idxs])
    hy = np.concatenate([TMS.TrueHitY[i] for i in idxs])
    hz = np.concatenate([TMS.TrueHitZ[i] for i in idxs])
    return np.stack([hx, hy, hz], axis=1)


def find_tms_slices_for_vertex(TMS, spill, vtx_id):
    """
    Truth_Info slice indices (in this spill) relevant to a TMS VertexID:
    slices where it's the dominant vertex (VertexIdOfMostEnergyInEvent), or
    where it appears as the vertex of a reconstructed track's primary
    particle (RecoTrackPrimaryParticleVtxId) -- covers subdominant/pileup
    vertices that still contributed a reco'd track to someone else's slice.
    """
    spill_idxs = np.where(TMS.SpillNo == spill)[0]
    slices = []
    for i in spill_idxs:
        if TMS.VertexIdOfMostEnergyInEvent[i] == vtx_id or vtx_id in TMS.RecoTrackPrimaryParticleVtxID[i]:
            slices.append(i)
    return slices


def build_tms_vertex_hits(TMS, spill, vtx_id):
    """
    True TMS hit positions (mm) restricted to the Truth_Info slice(s) that
    actually belong to this vertex, instead of the whole spill. Narrower
    (a few hundred hits instead of tens of thousands) AND more accurate:
    checked against a real muon, searching the whole spill picked up 59
    hits within 100mm of its polyline (true NHits=54, so 5+ were
    coincidental pileup hits from unrelated vertices); scoping to just this
    vertex's slice brought that down to 52 -- closer to truth because the
    unrelated-vertex false positives are excluded before the distance cut
    ever runs.
    """
    slices = find_tms_slices_for_vertex(TMS, spill, vtx_id)
    if not slices:
        return np.empty((0, 3))
    hx = np.concatenate([TMS.TrueHitX[i] for i in slices])
    hy = np.concatenate([TMS.TrueHitY[i] for i in slices])
    hz = np.concatenate([TMS.TrueHitZ[i] for i in slices])
    return np.stack([hx, hy, hz], axis=1)


def build_spine_vertex_table(truth_interactions):
    return [
        {"local_id": int(ixn.id), "orig_id": int(ixn.orig_id),
         "pos": np.asarray(ixn.vertex, dtype=float) * 10.0}   # cm -> mm
        for ixn in truth_interactions
    ]


def build_spine_particle_table(truth_particles):
    particles_by_interaction = {}
    for p in truth_particles:
        iid = int(p.interaction_id)  # local id, matches ixn.id (NOT ixn.orig_id)
        particles_by_interaction.setdefault(iid, []).append({
            "track_id": int(p.track_id),
            "pdg": int(p.pdg_code),
            "birth_pos": np.asarray(p.position, dtype=float) * 10.0,      # cm -> mm
            "end_pos": np.asarray(p.end_position, dtype=float) * 10.0,
        })
    return particles_by_interaction


def match_vertices(tms_vertices, spine_vertices, tol_mm):
    """Greedy nearest-neighbor bipartite match on vertex position."""
    pairs = []
    for i, tv in enumerate(tms_vertices):
        for j, sv in enumerate(spine_vertices):
            d = np.linalg.norm(tv["pos"] - sv["pos"])
            if d <= tol_mm:
                pairs.append((d, i, j))
    pairs.sort(key=lambda x: x[0])

    used_tms, used_spine = set(), set()
    matches = []
    for d, i, j in pairs:
        if i in used_tms or j in used_spine:
            continue
        used_tms.add(i)
        used_spine.add(j)
        matches.append({
            "tms_vtx_id": tms_vertices[i]["vtx_id"],
            "spine_local_id": spine_vertices[j]["local_id"],
            "spine_orig_id": spine_vertices[j]["orig_id"],
            "dist_mm": d,
        })
    return matches


def match_tracks(tms_particles, spine_particles, pos_tol_mm):
    """
    TrackId-first candidate join, confirmed with PDG + birth position.
    Returns confirmed matches, rejected id-collisions, and unmatched
    particles on each side.
    """
    tms_by_id = {p["track_id"]: p for p in tms_particles}
    spine_by_id = {p["track_id"]: p for p in spine_particles}

    candidate_ids = set(tms_by_id) & set(spine_by_id)

    confirmed, rejected = [], []
    for tid in sorted(candidate_ids):
        t, s = tms_by_id[tid], spine_by_id[tid]
        pdg_ok = (t["pdg"] == s["pdg"])
        pos_residual = float(np.linalg.norm(t["birth_pos"] - s["birth_pos"]))
        record = {"track_id": tid, "pdg": t["pdg"], "tms": t, "spine": s,
                  "pos_residual_mm": pos_residual}
        if pdg_ok and pos_residual <= pos_tol_mm:
            confirmed.append(record)
        else:
            rejected.append(record)

    unmatched_tms = sorted(set(tms_by_id) - candidate_ids)
    unmatched_spine = sorted(set(spine_by_id) - candidate_ids)
    return confirmed, rejected, unmatched_tms, unmatched_spine

In [ ]:
tms_vertices = build_tms_vertex_table(TMS, spill)
tms_particles_by_vtx = build_tms_particle_table(TMS, spill)
spine_vertices = build_spine_vertex_table(truth_interactions)
spine_particles_by_ixn = build_spine_particle_table(truth_particles)

# For scale reference only -- plot_matched_tracks() below scopes its own hit
# search per vertex via build_tms_vertex_hits(), not this whole-spill set.
spill_hits = build_tms_spill_hits(TMS, spill)
print(f"Nof true TMS hits (Truth_Info) in spill {spill}: {len(spill_hits)}")

vertex_matches = match_vertices(tms_vertices, spine_vertices, VERTEX_TOL_MM)

# NDLAr_idx == truth_interactions list index == ixn.id == spine_local_id
matched_tracks_by_ndlar_idx = {}
summary_rows = []
for m in vertex_matches:
    tms_p = tms_particles_by_vtx.get(m["tms_vtx_id"], [])
    spine_p = spine_particles_by_ixn.get(m["spine_local_id"], [])
    confirmed, rejected, unmatched_tms, unmatched_spine = match_tracks(
        tms_p, spine_p, TRACK_POS_TOL_MM)

    matched_tracks_by_ndlar_idx[m["spine_local_id"]] = {
        "tms_vtx_id": m["tms_vtx_id"],
        "vtx_dist_mm": m["dist_mm"],
        "confirmed": confirmed,
        "rejected": rejected,
    }
    summary_rows.append({
        "NDLAr_idx": m["spine_local_id"],
        "TMS_VertexID": m["tms_vtx_id"],
        "vtx_dist_mm": round(m["dist_mm"], 4),
        "n_confirmed_tracks": len(confirmed),
        "n_rejected_id_collisions": len(rejected),
    })

match_summary = pd.DataFrame(summary_rows).sort_values("NDLAr_idx").reset_index(drop=True)
print(f"Matched {len(vertex_matches)} / {len(tms_vertices)} TMS vertices to "
      f"{len(vertex_matches)} / {len(spine_vertices)} SPINE interactions "
      f"(tol={VERTEX_TOL_MM} mm)")
match_summary

# Setup Figure

In [ ]:
def setup_figure(figure, NDLAr_idx):
    figure.update_layout(
        width=1600,
        height=900,
        scene=dict(
            xaxis=dict(title="x [mm]", range=[-4500, 4500]),
            yaxis=dict(title="y [mm]", range=[-4000, 1000]),
            zaxis=dict(title="z [mm]", range=[0, 20000]),
            camera=dict(
                eye=dict(x=2.0, y=4.0, z=0.0),
                up=dict(x=0, y=-1, z=0)           # define Y as "up"
            ),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=4)
        ),

        margin=dict(l=0, r=0, b=0, t=50),

        showlegend=True,
        legend=dict(
            x=0.02,          # Places legend 2% from the left edge
            y=0.98,          # Places legend 2% from the top edge
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.8)', # Semi-transparent white background
            bordercolor='Black',
            borderwidth=1,
            itemsizing='constant'
        ),

        title={
            'text': f"ND-LAr Interaction {NDLAr_idx}",
            'x': 0.5,            # Centers the title
            'xanchor': 'center'
        }
    )

    figure.data = []  # Clear all traces

    plot_box(figure, -3730, 3730, -3702.23, 997.77, 11177.5, 18502.5, "blue", "TMS detector")
    plot_box(figure, -3478.48, 3478.48, -2166.71, 829.282, 4179.24, 9135.88, "black", "ND-LAr detector")

## Plot ND-LAr event by id

In [ ]:
def plot_NDLAr_interaction(figure, idx):
    vtx_x = truth_interactions[idx].vertex[0] * 10
    vtx_y = truth_interactions[idx].vertex[1] * 10
    vtx_z = truth_interactions[idx].vertex[2] * 10
    vtx_t = truth_interactions[idx].t

    figure.add_trace(
        go.Scatter3d(
            x=truth_interactions[idx].points[:, 0]*10,
            y=truth_interactions[idx].points[:, 1]*10,
            z=truth_interactions[idx].points[:, 2]*10,
            mode="markers",
            marker=dict(size=2,
                    color=NDLAr_colour,
                    symbol='circle'
            ),
            name=f"(#{idx}) ND-LAr Hits"
        )
    )

    figure.add_trace(
        go.Scatter3d(
            x = [vtx_x],
            y = [vtx_y],
            z = [vtx_z],
            mode="markers",
            marker=dict(size=4,
                    color=NDLAr_colour,
                    symbol='square-open'
            ),
            name=f"(#{idx}) ND-LAr Vertex [{vtx_x:.3f}, {vtx_y:.3f}, {vtx_z:.3f}, {vtx_t:.3f}]"
        )
    )

## Plot matched TMS tracks and hits

Draws TMS truth tracks whose `TrackId` candidate was CONFIRMED against the SPINE
truth particle for this ND-LAr interaction (same PDG + birth position within
`TRACK_POS_TOL_MM`) -- each as its true Birth -> Death segment, labeled with its
`TrackId`/PDG. Any rejected id-collision candidates are reported (not plotted).

True TMS hits (`Truth_Info` `TrueHitX/Y/Z`) are pulled only from the `Truth_Info`
slice(s) belonging to this vertex's `VertexID` (see point 4 above), then coloured
by nearest confirmed track (point 3 above) -- hits closer than `HIT_ASSIGN_TOL_MM`
to a confirmed track's polyline are drawn in that track's colour; every other hit
in the same slice(s) is drawn as small, low-opacity background points for context.

In [ ]:
def assign_hits_to_tracks(hits, confirmed, tol_mm):
    """
    Assign hits to confirmed tracks using each track's `TrueNHits` (Truth_Spill)
    as a quota, not a single fixed distance cut for every track. TrueNHits is
    only a ground-truth COUNT -- there's no branch anywhere giving the actual
    hit indices for a track -- so it can't tell us *which* hits are whose. But
    it tells us exactly *how many* each confirmed track should end up with,
    which is more informative than an arbitrary "assign the closest 100mm"
    rule.

    Algorithm: compute distance from every hit to every confirmed track's
    polyline (only pairs within tol_mm are candidates at all), sort all
    (hit, track) pairs globally by distance, then walk that list assigning
    each hit to its closest still-available track, decrementing that track's
    remaining TrueNHits quota. This matters most when two confirmed tracks'
    polylines pass near the same hits: a single global threshold has no way
    to divide contested hits correctly, while a quota lets each track claim
    only as many of the contested hits as its own truth count allows, leaving
    the rest for whichever other track needs them.

    Returns a list (len == len(hits)) of indices into `confirmed`, or None
    where no confirmed track had both a usable polyline and remaining quota
    within tol_mm.
    """
    tracks = [(i, rec["tms"]["polyline"], rec["tms"]["n_hits"])
              for i, rec in enumerate(confirmed) if rec["tms"]["polyline"] and rec["tms"]["n_hits"] > 0]

    pairs = []
    for h_idx, h in enumerate(hits):
        for i, pts, _ in tracks:
            d = point_polyline_dist(h, pts)
            if d < tol_mm:
                pairs.append((d, h_idx, i))
    pairs.sort(key=lambda x: x[0])

    remaining_quota = {i: quota for i, _, quota in tracks}
    assignment = [None] * len(hits)
    assigned_hits = set()
    for d, h_idx, i in pairs:
        if h_idx in assigned_hits or remaining_quota[i] <= 0:
            continue
        assignment[h_idx] = i
        assigned_hits.add(h_idx)
        remaining_quota[i] -= 1
    return assignment


def plot_matched_tracks(figure, ndlar_idx):
    entry = matched_tracks_by_ndlar_idx.get(ndlar_idx)
    if entry is None:
        print(f"NDLAr_idx {ndlar_idx}: no matched TMS vertex within {VERTEX_TOL_MM} mm")
        return figure

    tms_vtx = next(v for v in tms_vertices if v["vtx_id"] == entry["tms_vtx_id"])

    figure.add_trace(go.Scatter3d(
        x=[tms_vtx["pos"][0]], y=[tms_vtx["pos"][1]], z=[tms_vtx["pos"][2]],
        mode="markers",
        marker=dict(size=5, color="black", symbol="cross"),
        name=(f"(#{entry['tms_vtx_id']}) TMS Truth_Spill Vertex "
              f"(vtx dist={entry['vtx_dist_mm']:.4f} mm)")
    ))

    confirmed = entry["confirmed"]
    for i, rec in enumerate(confirmed):
        birth = rec["tms"]["birth_pos"]
        death = rec["tms"]["death_pos"]
        colour = strong_colours[i % len(strong_colours)]
        figure.add_trace(go.Scatter3d(
            x=[birth[0], death[0]], y=[birth[1], death[1]], z=[birth[2], death[2]],
            mode="lines+markers",
            line=dict(color=colour, width=4),
            marker=dict(size=3, color=colour),
            name=f"TrackId {rec['track_id']} (PDG {rec['pdg']}) matched TMS<->SPINE"
        ))

    if entry["rejected"]:
        print(f"NDLAr_idx {ndlar_idx}: {len(entry['rejected'])} candidate TrackId(s) "
              f"rejected as id collisions (not plotted): "
              f"{[r['track_id'] for r in entry['rejected']]}")

    # Only search hits from this vertex's own Truth_Info slice(s), not the whole
    # spill -- narrower and less prone to picking up coincidentally-nearby hits
    # from unrelated pileup vertices (see matching markdown, point 4).
    vertex_hits = build_tms_vertex_hits(TMS, spill, entry["tms_vtx_id"])

    if len(vertex_hits) and confirmed:
        assignment = assign_hits_to_tracks(vertex_hits, confirmed, HIT_ASSIGN_TOL_MM)

        for i, rec in enumerate(confirmed):
            sel = np.array([a == i for a in assignment])
            if not np.any(sel):
                continue
            colour = strong_colours[i % len(strong_colours)]
            pts = vertex_hits[sel]
            figure.add_trace(go.Scatter3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                mode="markers",
                marker=dict(size=3, color=colour, symbol="circle", opacity=0.9),
                name=f"TrueHits matched to TrackId {rec['track_id']} (PDG {rec['pdg']})"
            ))

        sel_none = np.array([a is None for a in assignment])
        if np.any(sel_none):
            pts = vertex_hits[sel_none]
            figure.add_trace(go.Scatter3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                mode="markers",
                marker=dict(size=1.5, color="lightgray", symbol="circle", opacity=0.3),
                name=f"Other TrueHits in VertexID {entry['tms_vtx_id']} slice(s) (unmatched)"
            ))

    return figure

## Draw Figure

In [ ]:
@interact(NDLAr_idx=(0, len(truth_interactions)-1))
def update_plot(NDLAr_idx):
    fig = go.Figure()

    setup_figure(fig, NDLAr_idx)
    plot_NDLAr_interaction(fig, NDLAr_idx)
    plot_matched_tracks(fig, NDLAr_idx)

    fig.show()